In [0]:
-- ================================================================
-- Question 1
-- Mean and population standard deviation, 2013-2018 inclusive
-- PySpark implementation is the primary Gold implementation.
-- This is the equivalent Spark SQL alternative.
-- ================================================================

SELECT
    AVG(population) AS mean_population,
    STDDEV_POP(population) AS population_stddev
FROM adb_rearc_assignment_workspace.silver.silver_population_yearly_fact
WHERE year BETWEEN 2013 AND 2018;



-- ================================================================
-- Question 2
-- Best year(s) per series based on sum of Q01-Q04 values.
-- Preserves genuine ties and identifies the latest tied year.
-- ================================================================

WITH yearly_sum AS (

    SELECT
        series_id,
        year,
        SUM(value) AS yearly_quarterly_sum
    FROM adb_rearc_assignment_workspace.silver.silver_bls_observations_fact
    WHERE period IN ('Q01', 'Q02', 'Q03', 'Q04')
    GROUP BY
        series_id,
        year
),

ranked AS (

    SELECT
        series_id,
        year,
        yearly_quarterly_sum,

        DENSE_RANK() OVER (
            PARTITION BY series_id
            ORDER BY yearly_quarterly_sum DESC
        ) AS value_rank

    FROM yearly_sum
),

best_years AS (

    SELECT
        series_id,
        year,
        yearly_quarterly_sum
    FROM ranked
    WHERE value_rank = 1
),

best_years_with_latest_flag AS (

    SELECT
        series_id,
        year,
        yearly_quarterly_sum,

        ROW_NUMBER() OVER (
            PARTITION BY series_id
            ORDER BY year DESC
        ) AS tie_year_rank

    FROM best_years
)

SELECT
    best.series_id,
    best.year AS best_year,
    ROUND(best.yearly_quarterly_sum, 3) AS yearly_quarterly_sum,

    CASE
        WHEN best.tie_year_rank = 1 THEN TRUE
        ELSE FALSE
    END AS is_latest_best_year,

    sector.sector_name,
    class.class_text,
    measure.measure_text,
    duration.duration_text,
    seasonal.seasonal_text,

    CONCAT_WS(
        ' | ',
        sector.sector_name,
        class.class_text,
        measure.measure_text,
        duration.duration_text,
        seasonal.seasonal_text
    ) AS series_description

FROM best_years_with_latest_flag best

LEFT JOIN adb_rearc_assignment_workspace.silver.silver_bls_series_dim series
    ON best.series_id = series.series_id

LEFT JOIN adb_rearc_assignment_workspace.silver.silver_bls_sector_dim sector
    ON series.sector_code = sector.sector_code

LEFT JOIN adb_rearc_assignment_workspace.silver.silver_bls_class_dim class
    ON series.class_code = class.class_code

LEFT JOIN adb_rearc_assignment_workspace.silver.silver_bls_measure_dim measure
    ON series.measure_code = measure.measure_code

LEFT JOIN adb_rearc_assignment_workspace.silver.silver_bls_duration_dim duration
    ON series.duration_code = duration.duration_code

LEFT JOIN adb_rearc_assignment_workspace.silver.silver_bls_seasonal_dim seasonal
    ON series.seasonal_code = seasonal.seasonal_code;



-- ================================================================
-- Question 3
-- PRS30006032 + Q01 joined to population by year where available
-- ================================================================

SELECT
    bls.year,
    bls.series_id,
    bls.period,
    bls.value AS bls_value,
    pop.population

FROM adb_rearc_assignment_workspace.silver.silver_bls_observations_fact bls

LEFT JOIN adb_rearc_assignment_workspace.silver.silver_population_yearly_fact pop
    ON bls.year = pop.year

WHERE
    bls.series_id = 'PRS30006032'
    AND bls.period = 'Q01'
ORDER BY bls.year;